# Forms and Input Validation

In this lesson, you will learn to validate form submissions before saving state and provide consistent feedback across input methods.

CSC-239 · Module 12 · Lesson 4 of 4

An editable field contains a proposed request. We will build a form that checks that request, explains rejection, and preserves the last accepted model value. Use the [module glossary](terms.md) to revisit terms after their explanations.


## Learning Goals

- Build a labeled JavaFX form that validates input before changing its model.
- Test valid, boundary, malformed, and repeated submissions with pointer and keyboard input.


## Why This Matters

A form connects a person's proposed request to stored application state. People may leave an entry blank, type a word where a number belongs, or enter a number outside the accepted range. An application needs to explain those problems while protecting the last valid state.

The previous lesson connected user actions to a guarded model and refreshed the view. A form adds editable text to that relationship. Checking a candidate before saving keeps a rejected request from replacing an accepted one. Sharing the submission behavior also keeps mouse and keyboard users on the same path, so the rules do not depend on how someone activates the form.


## Check Your Starting Point

Connect the earlier parsing and exception ideas with the model, handler, and view from the preceding lesson. Record the relationships before opening the answer.


Explain `Integer.parseInt`, `trim`, `NumberFormatException`, and a matching catch block. Explain how checking a candidate before assigning a private field protects saved state. Recall when a registered handler runs and how its model and Label have different roles.


In [ ]:
Parsing and exception roles:
Your response

Candidate before field assignment:
Your response

Handler/model/view roles:
Your response


<details>
<summary>Show answer</summary>

`Integer.parseInt` attempts to convert a String to an int. Text that cannot represent an int produces `NumberFormatException`; a matching catch block can respond to that failure. `trim` returns a String with surrounding characters up to U+0020 removed. It does not change the original String or remove arbitrary characters from its middle.

A candidate local variable holds a proposed value. Checking it before assigning a private field lets a method reject a request while retaining the previous field value. Registering an event handler supplies behavior for a later action. The model owns the state, and the handler refreshes the view from the result of the model operation.

</details>


## Video Demonstration

Watch a valid submission, a rejected edit, and recovery through Enter in the field. Predict the feedback and the saved model value separately. Compare the visible result with the model checks in this notebook.

<video controls preload="metadata" width="960">
  <source src="media/04_forms_and_input_validation/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/04_forms_and_input_validation/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the forms and input validation demonstration transcript](media/04_forms_and_input_validation/transcript.md).


## Concept

### Keep the proposed text separate from the saved quantity

A campus supply request needs a quantity from one to five items. A staff member should be able to edit the request, submit it, and receive a useful message when it cannot be accepted. A rejected edit must not replace the last accepted quantity.

A **TextField** is a single-line editable text control. The example begins with:

```java
    TextField input = new TextField("2");
```

The constructor creates the input control and puts the string `"2"` in it. Even though that text contains a digit, the field holds a String, not an int. The user can replace it with other text before requesting a save.

The **saved model state** is different from the field's proposed text. `QuantityModel` begins with its saved quantity at zero, meaning nothing has been saved in this session. Zero is an initial marker, not an accepted submitted quantity. The initial field can therefore show 2 while the model still stores zero and the status says Nothing saved.

In this example, saving means changing the model object used by this window. It does not write a file or make the value permanent across complete reruns. A fresh model starts with its own initial state. This distinction lets us reason separately about text being edited, a request being accepted, and the value currently stored.


### Read the current field when the user submits

A **form submission** is a deliberate request to read the current input and try to apply it. The form provides a Save quantity button:

```java
    Button save = new Button("Save quantity");
```

Creating that button does not save anything, just as creating Reserve one did not reserve a seat. Editing the field also does not automatically change the quantity. This form waits for the submission action before asking the model to accept a value.

The method `input.getText()` reads the field's current String. It belongs inside the registered submission behavior so each activation reads the user's latest edit. Reading it once while constructing the window would capture the initial text instead of the later request.

The model call `model.save(input.getText())` therefore has a clear order: obtain the current field text, then pass that text to the model for checking. The method returns a message describing what happened. The handler uses that message to update the visible status, as explained below.

The completed form also supports Enter while the input field has focus. Both the button and field will use the same submission behavior. The method of activation can differ; the rule for accepting a quantity should not.


### Check a candidate before changing accepted state

**Validate before mutation** means checking a proposed value before changing stored state. Mutation is a change to that state. A **candidate value** is a possible replacement that has not yet passed all of the application's rules.

The complete model class uses ordinary Java and does not depend on any control:

```java
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
```

The constructor sets the initial marker to zero, and the getter returns the current saved quantity. The `save` method receives a String and returns a String message. Its `try` block first evaluates `text.trim()`, which removes the surrounding ordinary spaces used in our examples. It produces text for the conversion; it does not edit the TextField or change the saved quantity.

`Integer.parseInt` then attempts to convert that text to an int. The local variable `candidate` receives the parsed number. A successful parse only establishes that the text represents an int; it does not establish that the application accepts that quantity.

The `if` uses `||`, the previously taught logical-or operator, to reject either a value below one or a value above five. Its early `return` sends back the range message before the assignment to `quantity`. Only a candidate that passes the range check reaches `quantity = candidate`.

If conversion cannot produce an int, it reports `NumberFormatException`. The matching catch returns the whole-number message. Text containing letters, empty text, or a number too large for int cannot pass that conversion. A number that parses but lies outside the allowed range takes the earlier range-rejection path instead.

Both rejection paths leave the saved field unchanged. The assignment occurs only after conversion and range checking succeed. Keeping the temporary candidate separate prevents a failed request from overwriting valid state or requiring the program to reconstruct an earlier value.


### Follow parsing, rejection, and recovery

<details class="animation-panel">
<summary>Show or hide the animation</summary>
<p><img src="media/04_forms_and_input_validation/parse_reject_recover.gif" alt="The ordinary model accepts 2, rejects six without mutation, and trims and accepts 5; getter results confirm both saved values." width="960" style="max-width:100%;height:auto;"></p>
</details>
<p>This loop lasts about 12.5 seconds.</p>
<p>A submission reads the current field String. Parsing can fail before the range check runs. A later valid proposal follows the same route and can still be accepted.</p>
<p><a href="media/04_forms_and_input_validation/parse_reject_recover_still.png">View the final state as a still image</a>.</p>


### Show what happened and let the user recover

**Visible validation feedback** tells the user whether a request was accepted and, if it was rejected, what kind of correction is needed. The form begins with a wrapping status label:

```java
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
```

The initial sentence reports that no submission has been accepted. Wrapping lets a longer correction message use another line when the layout gives the label less width. The message should remain readable without depending on color.

The model returns one of three kinds of feedback. An accepted quantity returns a Saved message containing its new value. A parsed number outside the permitted range returns Use a quantity from 1 to 5. Text that cannot be converted returns Enter a whole number from 1 to 5. These messages describe different outcomes instead of treating every rejection as the same problem.

For example, submitting the initial text 2 can save two. A later request that cannot be accepted changes the status message but leaves that saved two intact. The edited text can remain in the field so the user can correct it. Field text, feedback, and saved state therefore need not display the same information after an invalid request.

A later valid submission can replace the saved value and show a success message again. Rejection does not permanently disable the form. Test a valid request, a rejection, and a recovery in one continuing sequence so the relationship between them is visible.

A screenshot can show a correction message, but that message alone does not inspect the model's private field. The ordinary model checks use the getter after submissions to verify that rejection preserves the last accepted value. Native form tests separately check the displayed text and usable input paths.


### Keep a candidate separate from accepted state

<details class="animation-panel">
<summary>Show or hide the animation</summary>
<p><img src="media/04_forms_and_input_validation/candidate_vs_saved_state.gif" alt="Editing field text does not save it; a rejected submission preserves the last accepted quantity, which a separate getter check establishes." width="960" style="max-width:100%;height:auto;"></p>
</details>
<p>This loop lasts about 12.5 seconds.</p>
<p>The candidate is a local proposal. A rejection leaves the saved field unchanged, even though the handler displays a new correction message.</p>
<p><a href="media/04_forms_and_input_validation/candidate_vs_saved_state_still.png">View the final state as a still image</a>.</p>


### Give the field a persistent label and record its connection

The form places a descriptive label beside the editable field:

```java
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("2");
    fieldLabel.setLabelFor(input);
```

The label describes both the meaning of the entry and its accepted range. Unlike the editable value, that description remains visible when the user changes or removes the field text. A user should not have to preserve an example value merely to remember what belongs in the field.

**Control label association** connects a label to the control it describes. Calling `fieldLabel.setLabelFor(input)` records that relationship in JavaFX. Placing two nodes near each other is a visual arrangement; setting the association supplies a programmatic connection as well.

The call does not create the TextField, copy the label's words into it, or validate its contents. Construction, association, and validation still have separate roles. The field stores proposed text, its label explains the request, and the model checks the submitted value.

This relationship complements the layout work from the previous lesson. The VBox arranges the label, input, button, and status in order. Their placement should make sense visually, while their associations and keyboard behavior help the form remain understandable through other ways of interacting.


### Give button and Enter submissions the same rule

A **shared submission handler** is one handler used by more than one submission control. Our complete form defines and registers it with:

```java
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
```

These statements assume the model, status label, field, and button have been constructed. **`EventHandler<ActionEvent>`** is the type of the handler variable. `EventHandler` is JavaFX's functional interface for handling an event; the type argument `ActionEvent` identifies the kind of event this handler accepts. This combines the interface, generic-type, and lambda syntax taught earlier in the course.

The lambda's parameter `event` receives the notification when the handler is called. Its body first reads `input.getText()`, then calls `model.save` with that current String, and finally passes the returned message to `status.setText`. The expression returns no separate saved result to the control; the model applies the rule and the label displays its message.

The declaration creates that behavior without submitting the form. The next line registers the same handler on the Save quantity button. The final line registers it on the TextField, whose action is activated by Enter while the field has focus. Each later activation reads the field again; the handler does not keep the initial text as a saved request.

A focused button can also be activated with Space, as taught in the preceding lesson. Focus matters: Space while editing the field enters a space rather than acting as the button's activation. The form's keyboard tests must identify which control has focus before each action.

Sharing the handler keeps the validation and feedback path consistent. Separate copies could drift so that a button saved one value while Enter used an old value or applied different checks. This form registers both routes explicitly and keeps their decision in one model method. The complete worked program now combines editable text, validation, feedback, label association, and those consistent submission routes.


### Converge on one submission behavior

<details class="animation-panel">
<summary>Show or hide the animation</summary>
<p><img src="media/04_forms_and_input_validation/shared_submission_paths.gif" alt="Button and field Enter actions sequentially invoke the same handler with current text; both registrations alone leave initial state unchanged." width="960" style="max-width:100%;height:auto;"></p>
</details>
<p>This loop lasts about 12.5 seconds.</p>
<p>Button activation and Enter in the field both call the registered submit handler. Each call reads the current field text and uses the same model rule and feedback path.</p>
<p><a href="media/04_forms_and_input_validation/shared_submission_paths_still.png">View the final state as a still image</a>.</p>


### Prepare the supplied notebook support

This is supplied course support for running JavaFX inside IJava. Run it once after starting or restarting this notebook's Java kernel. The message `FX ready` means the support has initialized JavaFX and completed an operation on its application thread. Open the Workspace **Desktop** view to see the windows created by later cells.

`Fx.run(() -> { ... })` performs the enclosed UI work on the JavaFX Application Thread and waits for that short operation to finish. Use it for reading as well as changing a live window or its controls. `Fx.closeWindows()` hides the windows created by this kernel before another example opens its own. `Fx.start()` is safe to call again; it keeps JavaFX available after the last window closes.

The implementation below is provided runtime support. You do not need to write its thread-coordination machinery for this lesson. A thread is one sequence of execution; Module 13 studies how to coordinate more than one. Here your responsibility is to use the documented support operations and keep UI work short. The support uses a completion signal, a time limit, and an error holder so a later cell does not silently continue after unfinished or failed UI work.

Do not call `Platform.exit()` during notebook practice. That ends the toolkit for this kernel; restart the kernel and rerun setup if you do so. Closing a window is different from ending the toolkit. If setup reports a display error, check that the Workspace Desktop is running, then restart the kernel and rerun setup. A JavaFX window appears in the Desktop, not as an inline notebook control.


In [ ]:
import javafx.application.Platform;
import javafx.stage.Window;
import java.util.ArrayList;
import java.util.concurrent.CountDownLatch;
import java.util.concurrent.TimeUnit;
import java.util.concurrent.atomic.AtomicReference;
class Fx {
    static void run(Runnable action) throws InterruptedException {
        if (Platform.isFxApplicationThread()) {
            action.run();
            return;
        }
        CountDownLatch done = new CountDownLatch(1);
        AtomicReference<Throwable> failure = new AtomicReference<Throwable>();
        Platform.runLater(() -> {
            try { action.run(); }
            catch (Throwable error) { failure.set(error); }
            finally { done.countDown(); }
        });
        if (!done.await(10, TimeUnit.SECONDS)) {
            throw new IllegalStateException("FX operation timed out; restart the kernel.");
        }
        if (failure.get() != null) { throw new RuntimeException(failure.get()); }
    }
    static void start() throws InterruptedException {
        try { Platform.startup(() -> Platform.setImplicitExit(false)); }
        catch (IllegalStateException alreadyStarted) {
            // This call is also safe when this kernel already started JavaFX.
        }
        run(() -> Platform.setImplicitExit(false));
    }
    static void closeWindows() throws InterruptedException {
        run(() -> {
            for (Window window : new ArrayList<Window>(Window.getWindows())) {
                window.hide();
            }
        });
    }
}
Fx.start();
System.out.println("FX ready");


### Check the model without opening a window

An equipment desk accepts a whole-number quantity from one through five. The form will call its model with non-null field text. We can first inspect that model with a small ordinary Java program. This check uses no JavaFX controls and needs no `Fx` setup.

The constructor starts quantity at zero to mean nothing has been saved. That marker is different from an accepted submission. The `save` method trims and parses into a candidate, checks the range, and assigns the field only on success. Both rejection paths return a correction message before assignment.

Read how each call uses the same model. The first request saves two. The word is then rejected, and the getter checks that the earlier value remains. Finally, the padded request tests trimming and successful recovery. You may copy this complete example into a Java work cell.

```java
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
QuantityModel model = new QuantityModel();
System.out.println(model.save("2"));
System.out.println(model.save("six"));
System.out.println("Still saved: " + model.getQuantity());
System.out.println(model.save(" 5 "));
System.out.println("Now saved: " + model.getQuantity());
```

Expected output:

```text
Saved: 2
Enter a whole number from 1 to 5.
Still saved: 2
Saved: 5
Now saved: 5
```

The getter lines inspect saved state directly. A later form test will inspect something different: whether the field can be edited, each input route calls the handler, and the full feedback message appears. A correct model result does not by itself prove those interface behaviors.


## Worked Example

### Save an equipment quantity through a labeled form

A campus equipment desk needs a quantity from one through five. A person enters a proposal, deliberately submits it, and reads either confirmation or a correction message. A failed proposal must preserve the last accepted quantity. The same rule applies to the Save quantity button and Enter in the field.

**Construct the model and visible controls.** The imports provide the short JavaFX type names used below. Create a fresh QuantityModel, a persistent Quantity (1 to 5) label, and a TextField initially containing 2. Associate the label with the field. The wrapping status starts with Nothing saved., and Save quantity supplies the button action. Creating the field does not save its text.

**Connect one submission path.** Store a lambda in the `EventHandler<ActionEvent>` variable `submit`. When invoked, it reads the current field String, calls the model's `save` method, and displays the returned message. Register that same handler on the button and on the field. The model still owns parsing, range checks, and assignment; the handler connects those rules to visible feedback.

**Arrange, show, and test.** Put the label, field, button, and status in a VBox with gap 12 and padding 20. Attach it to a 420 by 280 Scene and show the titled Stage. The last line prints the initial model quantity. Later submissions change the model and view without rewriting that earlier construction report.


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("2");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Quantity Form");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});


Expected output:

```text
Saved quantity: 0
```

The initial quantity is zero and the status says Nothing saved. Activating Save quantity with the initial text 2 displays Saved: 2. A word produces the whole-number message; zero or six produces the range message. Each rejection leaves the saved two unchanged. Enter with 5 displays Saved: 5 and updates the saved value.


In the Workspace Desktop, inspect the persistent field label and the full feedback sentence. Select field text with Ctrl+A before replacing it. For an empty request, select the text and press Backspace. Keep deliberate surrounding spaces when testing trimming.

Use Enter while the TextField has focus. For a button's keyboard path, move focus to Save quantity with Tab and activate it with Space. Space while editing the field enters a space instead. Close the native window when finished; a new complete run creates a fresh model and controls. Use separate ordinary model checks to inspect saved quantity after rejection.


## Guided Practice

Start with a Supply Request form, then complete missing operations, change its proposed starting text, and repair an input path. Keep your original predictions beside actual results. The input sequences include endpoints, invalid values, and recovery because a single successful submission cannot establish the whole rule.

Use the supplied setup for complete GUI programs. Empty Java cells are places to write programs, not completed examples. Ordinary model checks need no window. Keep the GUI program and its separate model check available so you can compare visible feedback with saved state.


### Predict the initial Supply Request form

Read the complete program below before running it. Predict the initial console line, field text, status, and saved quantity separately. Then predict the status and saved quantity after submitting `4`, `0`, `word`, and ` 3 ` in that order. Explain whether typing alone saves a value. Record the prediction, then run the complete cell and inspect its initial state.


In [ ]:
Initial output, field, status, saved quantity:
Your response

4 / 0 / word / padded 3 predictions:
Your response

Editing versus saving:
Your response


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});


### Predict and perform the full submission sequence

Before each native action, record its predicted status and saved quantity in the corresponding response row. Submit these requests in the same window without rerunning between them: `4`, `0`, `6`, empty text, `six`, `2147483648`, ` 3 `, `1`, `5`, `word`, and `2`. Use the button for requests 1, 3, 5, 7, 9, and 11, and Enter in the field for requests 2, 4, 6, 8, and 10. Use pointer clicks for some button actions and Tab followed by Space on the focused button for at least one. Use Ctrl+A before replacement and Backspace for an empty request. Preserve deliberate surrounding spaces. Record each prediction before its action; keep the actual observations for the response area that follows.


In [ ]:
Predicted '4' status and saved quantity:
Your response

Predicted '0' status and saved quantity:
Your response

Predicted '6' status and saved quantity:
Your response

Predicted 'empty text' status and saved quantity:
Your response

Predicted 'six' status and saved quantity:
Your response

Predicted '2147483648' status and saved quantity:
Your response

Predicted ' 3 ' status and saved quantity:
Your response

Predicted '1' status and saved quantity:
Your response

Predicted '5' status and saved quantity:
Your response

Predicted 'word' status and saved quantity:
Your response

Predicted '2' status and saved quantity:
Your response


Record the full actual status and field text after every request in the sequence above. Close the native window with Alt+F4, rerun the complete program in the same kernel, inspect its fresh initial field and status, and close it again. Explain which observations show interface behavior and which saved-state claims still need getter evidence.


In [ ]:
Actual '4' field and full status:
Your response

Actual '0' field and full status:
Your response

Actual '6' field and full status:
Your response

Actual 'empty text' field and full status:
Your response

Actual 'six' field and full status:
Your response

Actual '2147483648' field and full status:
Your response

Actual ' 3 ' field and full status:
Your response

Actual '1' field and full status:
Your response

Actual '5' field and full status:
Your response

Actual 'word' field and full status:
Your response

Actual '2' field and full status:
Your response

Close, recreate initial state, final close:
Your response

Native observations versus getter evidence:
Your response


Trace a valid submission followed by `0`, blank text, and `2147483648`. For each, identify the current String, whether parsing succeeds, whether the range check rejects, whether quantity is assigned, and which message reaches the Label. Identify the label association and both registrations of the shared handler. Explain how an error message can appear while the earlier accepted quantity remains.


In [ ]:
Valid / zero / blank / oversized trace:
Your response

Label association and shared registrations:
Your response

Error feedback versus saved value:
Your response


<details>
<summary>Show answer</summary>

The new window is Supply Request, the field contains 4, and status begins as Nothing saved. The construction output is Saved quantity: 0. Creating a TextField does not submit its text. The first submission displays Saved: 4. Parsed 0 and 6 receive Use a quantity from 1 to 5.; empty text, six and 2147483648 receive Enter a whole number from 1 to 5. Those rejections do not assign quantity. The separate getter check verifies that four remains stored. The padded request ` 3 ` is trimmed and accepted, then the two endpoints 1 and 5 are accepted. After word is rejected, the final 2 is accepted. Both controls use the same submit object, so their submission rules agree.

The candidate is local: parsing and range checks happen before quantity is assigned. Zero reaches the range return; blank text and an oversized integer reach the catch. None of those paths assigns the field.

The handler then displays the returned message. `fieldLabel.setLabelFor(input)` associates the visible label with the field. Both `save.setOnAction(submit)` and `input.setOnAction(submit)` register the same handler.

The complete getter check below reports stored quantity after every request; native form actions separately verify the visible messages and input paths.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

Expected output:

```text
Saved quantity: 0
```

Common error: Treating the field's starting text as a value already saved. Treating the earlier notebook output as a live model display.

</details>


### Check the saved model after every request

Predict the returned message and stored quantity after each of the eleven requests in the complete ordinary Java check below. It constructs a fresh model without a window. Record all predictions before running the cell; this check complements the native observations rather than reading the state of the earlier window.


In [ ]:
Predicted '4' message and getter:
Your response

Predicted '0' message and getter:
Your response

Predicted '6' message and getter:
Your response

Predicted 'empty text' message and getter:
Your response

Predicted 'six' message and getter:
Your response

Predicted '2147483648' message and getter:
Your response

Predicted ' 3 ' message and getter:
Your response

Predicted '1' message and getter:
Your response

Predicted '5' message and getter:
Your response

Predicted 'word' message and getter:
Your response

Predicted '2' message and getter:
Your response


In [ ]:
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
QuantityModel model = new QuantityModel();
String[] requests = {"4", "0", "6", "", "six", "2147483648", " 3 ", "1", "5", "word", "2"};
for (String request : requests) {
    System.out.println(model.save(request));
    System.out.println("Stored quantity: " + model.getQuantity());
}


Record the actual returned message and getter value for every request. Keep your predictions and explain any correction. Explain why ordinary model checks and native feedback checks establish different parts of the form behavior.


In [ ]:
Actual '4' message and getter:
Your response

Actual '0' message and getter:
Your response

Actual '6' message and getter:
Your response

Actual 'empty text' message and getter:
Your response

Actual 'six' message and getter:
Your response

Actual '2147483648' message and getter:
Your response

Actual ' 3 ' message and getter:
Your response

Actual '1' message and getter:
Your response

Actual '5' message and getter:
Your response

Actual 'word' message and getter:
Your response

Actual '2' message and getter:
Your response

Corrections and two kinds of evidence:
Your response


<details>
<summary>Show answer</summary>

This uses the exact QuantityModel body from the form in a fresh model. Each request prints the returned message and then getQuantity. The stored value stays four through both range errors, blank input, a word and integer overflow. Padded three and both valid endpoints succeed. A later rejected word leaves five saved; the final two proves recovery. No window is created by this separate model check.

```java
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
QuantityModel model = new QuantityModel();
String[] requests = {"4", "0", "6", "", "six", "2147483648", " 3 ", "1", "5", "word", "2"};
for (String request : requests) {
    System.out.println(model.save(request));
    System.out.println("Stored quantity: " + model.getQuantity());
}
```

Expected output:

```text
Saved: 4
Stored quantity: 4
Use a quantity from 1 to 5.
Stored quantity: 4
Use a quantity from 1 to 5.
Stored quantity: 4
Enter a whole number from 1 to 5.
Stored quantity: 4
Enter a whole number from 1 to 5.
Stored quantity: 4
Enter a whole number from 1 to 5.
Stored quantity: 4
Saved: 3
Stored quantity: 3
Saved: 1
Stored quantity: 1
Saved: 5
Stored quantity: 5
Enter a whole number from 1 to 5.
Stored quantity: 5
Saved: 2
Stored quantity: 2
```

Common error: Treating the initial zero marker as an accepted submission. Replacing the model between requests and losing the preservation test.

</details>


<details>
<summary>Show answer</summary>
<details class="animation-panel">
<summary>Show or hide the animation</summary>
<p><img src="media/04_forms_and_input_validation/range_rejection_prefix.gif" alt="The first three requests 4, 0, and 6 accept 4 and reject both out-of-range values without replacing it; the full matrix continues beyond this prefix." width="960" style="max-width:100%;height:auto;"></p>
</details>
<p>This loop lasts about 12.5 seconds.</p>
<p>After four is accepted, the parsed zero and six requests return from the range check before assignment. The visible correction changes, but the getter still reports four. Blank text, a word, and an oversized integer fail parsing instead; neither rejection route assigns saved quantity.</p>
<p><a href="media/04_forms_and_input_validation/range_rejection_prefix_still.png">View the final state as a still image</a>.</p>
</details>


### Complete validation and shared submission

Replace all five uppercase markers in the displayed incomplete program. Reject a candidate outside 1 through 5 before assignment, associate the label with its field, and register the same submit handler for Enter and the button. Explain each replacement and why assignment follows the rejecting return. Predict the initial state and full eleven-request sequence before writing the complete solution in the Java work cell.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (__RANGE_CHECK__) {
                return "Use a quantity from 1 to 5.";
            }
            __SAVE_VALUE__
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    __ASSOCIATE_LABEL__
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = __SUBMIT_BODY__;
    save.setOnAction(submit);
    __ENTER_HANDLER__
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

After recording the plan, write and run the completed program. Repeat the eleven-request sequence and input methods above, then close, rerun the complete program to check fresh initial state, and close again.


In [ ]:
Five replacements and reasons:
Your response

Initial state and eleven-request predictions:
Your response


Record the actual initial report, full native request sequence, and close/recreate results. Keep the separate QuantityModel getter matrix as saved-state evidence and explain why it applies to your unchanged model body. Explain how assignment after the rejecting return protects accepted state.


In [ ]:
Initial output and actual eleven-request statuses:
Your response

Native input paths and lifecycle:
Your response

Separate getter evidence and assignment reasoning:
Your response


<details>
<summary>Show answer</summary>

Use the two range comparisons joined by ||. The rejecting branch returns before `quantity = candidate;`, so invalid candidates cannot replace saved state. Associate the label with input. The submit lambda reads the current field text, calls the model and updates status. Register that same handler for Enter in the field as well as the already supplied button registration.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

Expected output:

```text
Saved quantity: 0
```

Common error: Joining the outside-range comparisons with &&. Copying a stale field value before the action occurs. Constructing different validation behavior for the two input paths.

</details>


### Change proposed starting text without saving it

Preserve the original program. Change only the TextField starting String from `4` to ` 3 `, with one space on each side. Predict the initial report and status, then the results of Enter on the starting text, button submission of `word`, and Enter submission of `2`. Include both feedback and saved quantity. After recording the prediction, edit the complete program below and run it. Perform the three actions, then close, rerun to check its starting state, and close again.


In [ ]:
Initial state and three-action predictions:
Your response


In [ ]:
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});


Record the initial output, field text including spaces, and status. Record all three submission results and the close/recreate checks. Use the ordinary model check for saved-state evidence. Explain how trimming differs from changing the TextField or its model constructor.


In [ ]:
Actual initial and three-action results:
Your response

Close/recreate and saved-state evidence:
Your response

Trim explanation:
Your response


<details>
<summary>Show answer</summary>

Only the field's starting String changes. The fresh model still reports Saved quantity: 0 and status still says Nothing saved. Submitting the padded three displays Saved: 3 because trim removes its surrounding spaces before parsing. The word displays Enter a whole number from 1 to 5. without replacing three. The final 2 displays Saved: 2.

The field text is a proposed value, so merely displaying padded three does not save it.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField(" 3 ");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

Expected output:

```text
Saved quantity: 0
```

Common error: Changing the model constructor as well as the field text. Removing the spaces from the test before it reaches the TextField.

</details>


### Diagnose an Enter path that disagrees with the button

The displayed complete diagnostic has one faulty registration. Predict the status after Enter on the initial `4`, a button click, Enter after changing the field to `3`, and another button click. Explain the expected disagreement before running.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

After recording the prediction, copy this complete diagnostic into the Java work cell and run it. Perform those four actions in order, then close its native window.


In [ ]:
Enter 4 / button 4 / Enter 3 / button 3 predictions:
Your response

Registration diagnosis:
Your response


Record all four actual diagnostic statuses and confirm the window closed. Keep the original prediction. Explain what the difference between Enter and button results tells you about the model rule and the registration.


In [ ]:
Four actual diagnostic statuses:
Your response

Native close and explanation:
Your response


Identify the correct registration for the field and explain why it repairs the input path. Predict the full eleven-request quantity sequence for the repaired program before editing. Then repair the complete source in the Java work cell above, rerun it, and perform the same pointer, Enter, and focused-Space submission methods. Close the window, rerun to check fresh initial state, and close again.


In [ ]:
Repair registration and reason:
Your response

Repaired eleven-request predictions:
Your response


Record the repaired eleven-request statuses, keyboard and pointer observations, and close/recreate checks. Use the separate model getter matrix for the unchanged model rules. Explain why changing the numeric range would not fix the faulty registration.


In [ ]:
Actual repaired request statuses and input paths:
Your response

Getter comparison and native lifecycle:
Your response

Why range edits do not repair registration:
Your response


<details>
<summary>Show answer</summary>

The diagnostic has no action handler on the TextField. Enter on its initial 4 leaves Nothing saved.; clicking Save quantity displays Saved: 4. After editing the field to 3, Enter leaves Saved: 4; clicking the button displays Saved: 3. The model's range rule is unchanged.

Restore `input.setOnAction(submit);` so Enter and the button use the same submit behavior.

In the complete repaired program, Enter submits current field text and updates status, just as the pointer path does.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class QuantityModel {
    private int quantity;
    public QuantityModel() { quantity = 0; }
    public int getQuantity() { return quantity; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 5) {
                return "Use a quantity from 1 to 5.";
            }
            quantity = candidate;
            return "Saved: " + quantity;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 5.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    QuantityModel model = new QuantityModel();
    Label fieldLabel = new Label("Quantity (1 to 5)");
    TextField input = new TextField("4");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save quantity");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Supply Request");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved quantity: " + model.getQuantity());
});
```

Expected output:

```text
Saved quantity: 0
```

Common error: Changing QuantityModel.save even though the pointer path already uses the required rules. Checking the console construction line to decide whether Enter submitted.

</details>


## Independent Practice

Design a loan form for a campus equipment service. Apply the same separation between proposed text, validation feedback, and accepted model state to a different range. Plan the rules and input routes before building the complete program.


### Build the Loan Form

Initialize saved days to 0 to mean that nothing has been saved yet.

Create a LoanModel and Loan Form using a TextField initially 3, a Loan days (1 to 7) label associated with that field, a Save loan button, and visible status. Store only whole numbers from 1 through 7 after trimming whitespace. Return Saved: n on success, Use 1 to 7 days. for an out-of-range number, and Enter a whole number from 1 to 7. for nonnumeric input. Keep the previous saved days on any rejection. Use one `EventHandler<ActionEvent>` for button and Enter. Test 3, 1, 7, 0, 8, blank text, a word, an oversized integer, surrounding spaces and a valid submission after an error. Include all imports and fresh model/control objects in the complete program, using the supplied Fx setup for UI work. Start status with Nothing saved., use a 420 by 280 Scene, VBox gap 12, padding 20, and wrapping status text. The construction report is Saved days: followed by the model getter. Before opening the solution, write and run your own program, compare native feedback with your planned rules, and explain the label association, assignment placement and shared handler.

Plan the model, messages, label association, and shared handler. Predict the initial console report, proposed field text, status, and saved days. After recording the plan, write and run your complete program in the Java work cell.


In [ ]:
Model, messages, association, handler plan:
Your response

Initial output, field, status, saved days prediction:
Your response


Record the initial Loan Form output, field text, and visible status. Explain the label association, placement of the saved-state assignment, and shared handler. Separate what you saw in the window from what the model source establishes.


In [ ]:
Actual initial output, field and status:
Your response

Association, assignment, shared handler explanation:
Your response


### Test the Loan Form across rejection and recovery

Keep one complete GUI program and use a fresh window for this sequence: `3`, `1`, `7`, `0`, `8`, empty text, `word`, `2147483648`, ` 5 `, `word`, and `2`. Before each action, record the predicted feedback and saved days in the corresponding row. Alternate button and field-Enter submissions; include pointer clicks and at least one Tab-to-button followed by Space. Preserve the padded text and use Backspace for the empty request. Do not rerun between submissions.


In [ ]:
Predicted '3' feedback and saved days:
Your response

Predicted '1' feedback and saved days:
Your response

Predicted '7' feedback and saved days:
Your response

Predicted '0' feedback and saved days:
Your response

Predicted '8' feedback and saved days:
Your response

Predicted 'empty text' feedback and saved days:
Your response

Predicted 'word' feedback and saved days:
Your response

Predicted '2147483648' feedback and saved days:
Your response

Predicted ' 5 ' feedback and saved days:
Your response

Predicted 'word' feedback and saved days:
Your response

Predicted '2' feedback and saved days:
Your response


Record the full actual native status after every loan request. Close the window, rerun the complete GUI program to inspect fresh initial field and status, and close again. Explain what the successful final request adds to the preceding error tests.


In [ ]:
Actual '3' full native status:
Your response

Actual '1' full native status:
Your response

Actual '7' full native status:
Your response

Actual '0' full native status:
Your response

Actual '8' full native status:
Your response

Actual 'empty text' full native status:
Your response

Actual 'word' full native status:
Your response

Actual '2147483648' full native status:
Your response

Actual ' 5 ' full native status:
Your response

Actual 'word' full native status:
Your response

Actual '2' full native status:
Your response

Close/recreate and recovery explanation:
Your response


### Test saved days without a window

Predict each returned message and stored `getDays` value for the same eleven loan requests. Then write a separate complete ordinary Java test in the work cell below: include your full LoanModel class, a fresh model, and the exact request Strings in their listed order. Print the returned message and then `Stored days: ` followed by the getter after every request. Preserve the complete GUI program above. This test opens no window and needs no Fx setup. Run it only after recording the predictions.


In [ ]:
Predicted '3' model message and getter:
Your response

Predicted '1' model message and getter:
Your response

Predicted '7' model message and getter:
Your response

Predicted '0' model message and getter:
Your response

Predicted '8' model message and getter:
Your response

Predicted 'empty text' model message and getter:
Your response

Predicted 'word' model message and getter:
Your response

Predicted '2147483648' model message and getter:
Your response

Predicted ' 5 ' model message and getter:
Your response

Predicted 'word' model message and getter:
Your response

Predicted '2' model message and getter:
Your response


Record every actual message and getter pair. Explain which inputs fail parsing, which fail the range check, and why each rejection preserves the most recent accepted days. Explain the final recovery and why a screenshot of an error alone cannot prove saved state.


In [ ]:
Actual '3' model message and getter:
Your response

Actual '1' model message and getter:
Your response

Actual '7' model message and getter:
Your response

Actual '0' model message and getter:
Your response

Actual '8' model message and getter:
Your response

Actual 'empty text' model message and getter:
Your response

Actual 'word' model message and getter:
Your response

Actual '2147483648' model message and getter:
Your response

Actual ' 5 ' model message and getter:
Your response

Actual 'word' model message and getter:
Your response

Actual '2' model message and getter:
Your response

Parsing/range/recovery reasoning:
Your response

Model proof versus screenshot:
Your response


<details>
<summary>Show answer</summary>

LoanModel starts with zero as the nothing-saved marker. It trims and parses into candidate, checks 1 through 7, and assigns days only on success. Both pointer and Enter invoke one submit handler and display the returned message. The label is explicitly associated with input. The complete source below matches the required Loan Form. Test it with native actions, then use the separate ordinary getter matrix to verify that each rejected request leaves the last valid days unchanged.

```java
import javafx.stage.Stage;
import javafx.scene.Scene;
import javafx.scene.control.Label;
import javafx.scene.control.Button;
import javafx.scene.layout.VBox;
import javafx.geometry.Insets;
import javafx.scene.control.TextField;
import javafx.event.ActionEvent;
import javafx.event.EventHandler;
class LoanModel {
    private int days;
    public LoanModel() { days = 0; }
    public int getDays() { return days; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 7) {
                return "Use 1 to 7 days.";
            }
            days = candidate;
            return "Saved: " + days;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 7.";
        }
    }
}
Fx.closeWindows();
Fx.run(() -> {
    Stage stage = new Stage();
    LoanModel model = new LoanModel();
    Label fieldLabel = new Label("Loan days (1 to 7)");
    TextField input = new TextField("3");
    fieldLabel.setLabelFor(input);
    Label status = new Label("Nothing saved.");
    status.setWrapText(true);
    Button save = new Button("Save loan");
    EventHandler<ActionEvent> submit = event -> status.setText(model.save(input.getText()));
    save.setOnAction(submit);
    input.setOnAction(submit);
    VBox root = new VBox(12, fieldLabel, input, save, status);
    root.setPadding(new Insets(20));
    stage.setTitle("Loan Form");
    stage.setScene(new Scene(root, 420, 280));
    stage.show();
    System.out.println("Saved days: " + model.getDays());
});
```

Expected output:

```text
Saved days: 0
```

Common error: Accepting an invalid value by assigning days before the check. Using the quantity range instead of the loan range. Providing a visible label without its association to the TextField.

**Additional test: loan_model_matrix.** This separate complete program uses the exact LoanModel body and prints getDays after every save attempt. Successful 3, 1 and 7 establish each valid value. The range errors 0 and 8, blank text, word and oversized integer preserve seven. Padded five succeeds; the next word preserves five; the final two succeeds. The output proves saved-state behavior for this model sequence. The native form test separately verifies control association, field editing, pointer/keyboard paths and visible feedback.

```java
class LoanModel {
    private int days;
    public LoanModel() { days = 0; }
    public int getDays() { return days; }
    public String save(String text) {
        try {
            int candidate = Integer.parseInt(text.trim());
            if (candidate < 1 || candidate > 7) {
                return "Use 1 to 7 days.";
            }
            days = candidate;
            return "Saved: " + days;
        } catch (NumberFormatException error) {
            return "Enter a whole number from 1 to 7.";
        }
    }
}
LoanModel model = new LoanModel();
String[] requests = {"3", "1", "7", "0", "8", "", "word", "2147483648", " 5 ", "word", "2"};
for (String request : requests) {
    System.out.println(model.save(request));
    System.out.println("Stored days: " + model.getDays());
}
```

Expected output:

```text
Saved: 3
Stored days: 3
Saved: 1
Stored days: 1
Saved: 7
Stored days: 7
Use 1 to 7 days.
Stored days: 7
Use 1 to 7 days.
Stored days: 7
Enter a whole number from 1 to 7.
Stored days: 7
Enter a whole number from 1 to 7.
Stored days: 7
Enter a whole number from 1 to 7.
Stored days: 7
Saved: 5
Stored days: 5
Enter a whole number from 1 to 7.
Stored days: 5
Saved: 2
Stored days: 2
```

</details>


## Summary

A TextField stores proposed text. Submission reads its current value at a deliberate action; construction and editing alone do not save it. Parse into a candidate, check the allowed range, and only then replace saved model state.

Visible feedback explains either success or correction. An error message may replace the status text while the previous valid value remains saved. A persistent label and its association describe the field, while one shared handler keeps button and Enter submissions consistent.

Test valid entries, both endpoints, malformed text, out-of-range values, and recovery. Use native interaction to check the input paths and visible messages. Use an ordinary model check with a getter to establish the saved value after each request.


With answers closed, explain how field text, status text, and saved model state can differ. Trace a valid save, a rejected request, and a later successful recovery. Include the order of parsing, range checking, assignment, and displayed feedback.


In [ ]:
Field/status/saved-state distinctions:
Your response

Valid / rejected / recovered trace:
Your response


<details>
<summary>Show answer</summary>

The field contains the latest proposed String. The status contains the model's most recent response message. The saved field contains the last accepted numeric value. Those three values have different roles and can differ after a rejection.

A successful save parses and accepts the candidate before assigning it. A rejection returns from the range check or catch without assigning saved state. The handler still displays that rejection message. A later valid request can then replace the saved value and display confirmation, showing that the failure did not prevent recovery.

</details>


## Reflection

Transfer the form structure to another campus request. Include a clear rule, a visible correction, and equivalent submission paths. Explain how your test distinguishes a displayed error from proof that the earlier saved value survived.

Later course work connects background operations with interfaces. Keep the current form's callbacks short; coordinating slow or waiting work is a separate topic.


Design a small numeric form for another campus request. State its initial nothing-saved marker, accepted range, two distinct rejection messages, and keyboard submission path. Describe a test that proves an invalid edit preserves the last valid model value, and explain how that differs from checking visible feedback.


In [ ]:
Form rules and input paths:
Your response

Saved-state test and visible-feedback check:
Your response


## Supplemental Reading

- [JavaFX 21 TextField API](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/TextField.html) documents editable text and action events.
- [JavaFX 21 Label API](https://openjfx.io/javadoc/21/javafx.controls/javafx/scene/control/Label.html) documents labelFor association.
- [JavaFX 21 EventHandler API](https://openjfx.io/javadoc/21/javafx.base/javafx/event/EventHandler.html) defines the shared callback contract.
- [Java 21 Integer API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Integer.html#parseInt(java.lang.String)) documents integer parsing and parse failures.
- [Java 21 String API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/String.html#trim()) describes removal of surrounding characters through trim.
